<a href="https://colab.research.google.com/github/NikoriakViktor/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_2/lessons/lesson_21_inheritance_polymorphism/note_lesson_21_inheritance_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Урок 21 — Наслідування, поліморфізм: таксі, кур'єр, самовивіз

Сервіс «Смачно + Таксі» тепер має кілька способів доставки: **таксі** (ціна за районом), **кур'єр-пішохід** (40 грн, лише Поділ), **самовивіз** (безкоштовно), а вночі — надбавка. Звіт для всіх однаковий: `fare()`, `eta()`, `describe()`.

| Розділ | Що всередині | Приклади | Задача |
|---|---|---|---|
| 1 | базовий клас і нащадок, MRO, пошук атрибутів, `NotImplementedError` | 1.1–1.5 | |
| 2 | `super()`: розширити, а не замінити | 2.1–2.3 | |
| 3 | поліморфізм, `if` за типом, качина типізація | 3.1–3.6 | |
| 4 | міксини й множинне наслідування, естафета `super()` | 4.1–4.4 | **Задача 1** |
| 5 | пастки: несумісні сигнатури, нащадок, що ламає батька (LSP) | 5.1–5.4 | |
| 6 | розібраний приклад: звіт, що не знає типів доставок | 6.1–6.2 | **Задача 2** |
| 7 | архітектура: ієрархія чи композиція | 7.1–7.3 | **Задача 3** |

**Як працювати**

- Виконуй клітинки **зверху вниз**. Клітинка з визначенням (`class`, `def`) лише створює клас чи функцію — **запуск завжди в наступній клітинці**. Так приклад можна перезапускати, не перевизначаючи класи.
- Перед **🔮 Прогнозом** спершу відповідай сам, потім розгортай відповідь.
- Задач лише три; решта — розібрані приклади: прочитай, запусти, зміни щось і запусти знову.
- Теорія, діаграми класів і порівняння «ієрархія проти композиції» — у книзі: [Урок 21. Наслідування, поліморфізм](https://nikoriakviktor.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m2/lesson_21/).

## 🔁 Пригадай (без підглядання)

1. Де Python шукає `first.FEES`, якщо в екземплярі його немає?
2. Що повертає `isinstance(True, int)`?
3. Що таке фабрика функцій?

<details>
<summary>Відповіді</summary>

1. У класі.
2. `True`: `bool` — нащадок `int`.
3. Функція, що створює й повертає налаштовану функцію.

</details>

---
## 1. Базовий клас і перший нащадок

Усі доставки мають номер замовлення й район, і всі вміють описати себе. Це спільне — у **базовому** (батьківському) класі.

### Приклад 1.1. Базовий клас `Delivery`

In [ ]:
class Delivery:
    """Будь-яка доставка: номер замовлення, район, ціна, час."""

    kind = "Доставка"

    def __init__(self, order_id, district):
        self.order_id = order_id
        self.district = district

    def fare(self):
        raise NotImplementedError(f"{type(self).__name__} має визначити fare()")

    def eta(self):
        return 30

    def describe(self):
        return f"{self.kind} №{self.order_id}: {self.district}, {self.fare()} грн, ~{self.eta()} хв"

In [ ]:
print([name for name in Delivery.__dict__ if not name.startswith("__")])

plain = Delivery(1, "Поділ")
print(plain.__dict__, plain.eta())
try:
    plain.describe()
except NotImplementedError as error:
    print("NotImplementedError:", error)

- у класі лежать `kind`, `fare`, `eta`, `describe` (і `__init__`, який ми відфільтрували разом з іншими службовими іменами);
- `fare()` базовий клас не знає — ціна залежить від способу доставки. Тож він піднімає `NotImplementedError`: «нащадок, визнач мене». Сама по собі «просто доставка» опис дати не може;
- `eta()` має типове значення (30 хв), яке нащадок може змінити;
- `describe()` уже написаний і користується `fare()` та `eta()` — **якими б вони не були**.

### Приклад 1.2. Нащадок `Pickup`

**Нащадок** (підклас) вказує батька в дужках і отримує все, що в того є.

In [ ]:
class Pickup(Delivery):
    kind = "Самовивіз"

    def fare(self):
        return 0

    def eta(self):
        return 15

In [ ]:
pickup = Pickup(3, "кафе")
print(pickup.describe())
print(isinstance(pickup, Pickup), isinstance(pickup, Delivery), issubclass(Pickup, Delivery))

У `Pickup` немає ні `__init__`, ні `describe` — вони знайдені в `Delivery`. А `fare`, `eta` і `kind` — власні: нащадок їх **перевизначив**. Самовивіз **є** доставкою (is-a), тому `isinstance(pickup, Delivery)` — `True`; `issubclass` питає те саме про класи.

### Приклад 1.3. MRO нащадка

**🔮 Прогноз:** у якому порядку Python шукатиме метод для `Pickup`?

<details>
<summary>Відповідь</summary>

`Pickup → Delivery → object`. `describe` знайдено в `Delivery`, але `self.fare()` усередині знову шукається з `Pickup`.

</details>

In [ ]:
print([cls.__name__ for cls in Pickup.mro()])
print(Pickup.__bases__)

`object` в кінці ланцюжка — спільний предок усіх класів Python. Саме від нього кожен клас отримує, наприклад, типовий `__repr__`.

### Приклад 1.4. Де знайшовся кожен атрибут

Пройдемо MRO руками: для кожного імені знайдемо перший клас, у чиєму `__dict__` воно є.

In [ ]:
def where(obj, name):
    """Перший клас у MRO об'єкта, де лежить атрибут name."""
    if name in obj.__dict__:
        return "екземпляр"
    for cls in type(obj).mro():
        if name in cls.__dict__:
            return cls.__name__
    return "ніде"

In [ ]:
for name in ["order_id", "kind", "fare", "eta", "describe", "__init__", "__repr__", "driver"]:
    print(f"{name:9} → {where(pickup, name)}")

print(pickup.describe() == Delivery.describe(pickup))

```text
pickup.describe()
   ├─ в екземплярі?  ні
   ├─ у Pickup?      ні
   └─ у Delivery?    так → Delivery.describe(pickup)
                              └─ усередині self.fare(): пошук знову з початку → Pickup.fare → 0
```

Головне — останній крок: `describe` написаний у `Delivery`, але `self` — це `pickup`, і `self.fare()` знаходить `Pickup.fare`. Базовий клас задає **сценарій**, нащадки підставляють **деталі**. `__repr__` знайшовся аж в `object`, а `driver` — ніде: у самовивозу водія немає.

### Приклад 1.5. Нащадок забув перевизначити `fare()`

In [ ]:
class BikeDraft(Delivery):
    kind = "Велосипед"

    def eta(self):
        return 20

In [ ]:
bike = BikeDraft(4, "Поділ")
print(bike.eta())
try:
    print(bike.describe())
except NotImplementedError as error:
    print("NotImplementedError:", error)

`eta()` працює, а `describe()` падає з повідомленням, у якому **ім'я нащадка**: `type(self).__name__` у базовому класі дає клас конкретного об'єкта. Саме тому `fare()` у базі не повертає «якесь» число, а голосно падає: забуте перевизначення видно одразу, а не в звіті з неправильною сумою.

---
## 2. `super()`: розширити, а не замінити

Таксі потребує ще й водія. Новий `__init__` нащадка **заміняє** батьківський, тому спершу треба викликати батьківський через `super()`, а потім додати своє.

### Приклад 2.1. `TaxiDelivery`

In [ ]:
class TaxiDelivery(Delivery):
    kind = "Таксі"
    FEES = {"Поділ": 60, "Оболонь": 80, "Печерськ": 90}

    def __init__(self, order_id, district, driver):
        super().__init__(order_id, district)
        self.driver = driver

    def fare(self):
        return self.FEES[self.district]

    def eta(self):
        return 25

    def describe(self):
        return super().describe() + f", водій {self.driver}"

In [ ]:
taxi = TaxiDelivery(1, "Оболонь", "D-3")
print(taxi.describe())
print(taxi.__dict__)
print([cls.__name__ for cls in TaxiDelivery.mro()])
print([name for name in TaxiDelivery.__dict__ if not name.startswith("__")])

`super()` — це «наступний клас за MRO». `super().__init__(…)` заповнив `order_id` і `district`, потім таксі додало `driver` — у `__dict__` усі три. `super().describe()` зібрав спільний опис, до якого таксі дописало водія. Власні члени `TaxiDelivery` — `kind`, `FEES`, `fare`, `eta`, `describe` (плюс `__init__`).

### Приклад 2.2. Забули `super().__init__`

**🔮 Прогноз:** що станеться, якщо в `__init__` таксі не викликати батьківський?

<details>
<summary>Відповідь</summary>

Об'єкт створиться, але в ньому буде лише `driver`. Перший же `describe()` впаде з `AttributeError`: `order_id` і `district` записував батьківський `__init__`, який ніхто не викликав.

</details>

In [ ]:
class BrokenTaxi(TaxiDelivery):
    def __init__(self, order_id, district, driver):
        self.driver = driver              # ← super().__init__ забули

In [ ]:
broken = BrokenTaxi(2, "Поділ", "D-1")
print(broken.__dict__)
try:
    broken.describe()
except AttributeError as error:
    print("AttributeError:", error)

Помилка з'являється не там, де її зробили (`__init__`), а пізніше — у `describe`. Тому правило просте: **перевизначив `__init__` — першим рядком виклич `super().__init__(...)`**.

### Приклад 2.3. Замінити чи розширити метод

Два способи перевизначити `describe()`: написати заново або дописати до батьківського через `super()`.

In [ ]:
class ShortTaxi(TaxiDelivery):
    def describe(self):                   # заміна: батьківський опис не використовуємо
        return f"Таксі, водій {self.driver}"


class ExtendedTaxi(TaxiDelivery):
    def describe(self):                   # розширення: беремо готове й дописуємо
        return super().describe() + ", оплата карткою"

In [ ]:
print(ShortTaxi(5, "Печерськ", "D-4").describe())
print(ExtendedTaxi(5, "Печерськ", "D-4").describe())

Заміна втратила номер, район, ціну й час — і якщо `Delivery.describe` колись зміниться (скажімо, додасться валюта), `ShortTaxi` цього не побачить. Розширення через `super()` отримує всі майбутні зміни батька автоматично. Заміна доречна, лише коли поведінка батька справді не потрібна.

---
## 3. Поліморфізм: один виклик — різна поведінка

### Приклад 3.1. Кур'єр — третій спосіб

Кур'єр ходить лише Подолом. Правило району перевіряємо в `__init__` **до** виклику батьківського: неправильну доставку не створити.

In [ ]:
class CourierDelivery(Delivery):
    kind = "Кур'єр"
    AREA = {"Поділ"}

    def __init__(self, order_id, district):
        if district not in self.AREA:
            raise ValueError(f"кур'єр не ходить у район {district}")
        super().__init__(order_id, district)

    def fare(self):
        return 40

    def eta(self):
        return 45

In [ ]:
print(CourierDelivery(2, "Поділ").describe())
try:
    CourierDelivery(9, "Печерськ")
except ValueError as error:
    print("ValueError:", error)

### Приклад 3.2. Звіт, якому байдуже, що за доставка

In [ ]:
deliveries = [TaxiDelivery(1, "Оболонь", "D-3"), CourierDelivery(2, "Поділ"), Pickup(3, "кафе")]

for delivery in deliveries:
    print(delivery.describe())
print("Разом за доставку:", sum(delivery.fare() for delivery in deliveries), "грн")

Той самий рядок `delivery.describe()` робить три різні речі. Це **поліморфізм** («багато форм»): код звертається до спільного **інтерфейсу** — набору методів `fare`, `eta`, `describe`, — а кожен клас відповідає по-своєму. У циклі немає жодного `if isinstance(...)`.

### Приклад 3.3. Те саме без поліморфізму: `if` за типом

Для порівняння — як рахували б ціну, якби класи не мали власного `fare()`:

In [ ]:
FEES = {"Поділ": 60, "Оболонь": 80, "Печерськ": 90}


def fare_by_type(delivery):
    if isinstance(delivery, TaxiDelivery):
        return FEES[delivery.district]
    elif isinstance(delivery, CourierDelivery):
        return 40
    elif isinstance(delivery, Pickup):
        return 0
    raise TypeError(f"невідомий тип доставки: {type(delivery).__name__}")

In [ ]:
print([fare_by_type(delivery) for delivery in deliveries])
try:
    fare_by_type(BikeDraft(4, "Поділ"))
except TypeError as error:
    print("TypeError:", error)

Ціни ті самі, але знання про кожен тип тепер **розкидане**: нова доставка — нова гілка тут, і в `eta_by_type`, і в `describe_by_type`… Забудеш одну — `TypeError` у звіті. З поліморфізмом новий спосіб доставки — новий клас, а звіт не змінюється.

### Приклад 3.4. Качина типізація

Python не питає, **від кого** походить об'єкт, — лише чи **вміє** він потрібне. «Якщо воно ходить як качка і крякає як качка — це качка».

**🔮 Прогноз:** чи спрацює звіт з об'єктом, що **не** наслідує `Delivery`?

<details>
<summary>Відповідь</summary>

Так: звіт лише викликає `fare()` і `describe()`. `isinstance(..., Delivery)` при цьому — `False`.

</details>

In [ ]:
class PartnerCourier:
    """Клас з бібліотеки партнера — не наслідує Delivery."""

    def __init__(self, order_id, price):
        self.order_id = order_id
        self.price = price

    def fare(self):
        return self.price

    def describe(self):
        return f"Партнер №{self.order_id}: {self.price} грн"

In [ ]:
mixed = deliveries + [PartnerCourier(4, 95)]
print(sum(delivery.fare() for delivery in mixed))
print(mixed[-1].describe(), isinstance(mixed[-1], Delivery))

80 + 40 + 0 + 95 = 215. Наслідування — один спосіб отримати спільний інтерфейс, але не єдиний.

### Приклад 3.5. Качина типізація у вбудованих функціях

`len()` працює для рядків, списків і словників, хоча спільного предка з методом `__len__` у них немає. `len(x)` просто викликає `x.__len__()` — тож працює і з нашим класом, якщо він уміє `__len__` (спеціальні методи — урок 24).

In [ ]:
class Shift:
    """Зміна диспетчера: список доставок."""

    def __init__(self, deliveries):
        self.deliveries = list(deliveries)

    def __len__(self):
        return len(self.deliveries)

In [ ]:
shift_today = Shift(mixed)
print(len("Поділ"), len([540, 320]), len({"D-3": 2}), len(shift_today))
print(Shift.mro())

`Shift` наслідує лише `object`, але `len(shift_today)` — 4. Для `len` важить не родовід, а наявність методу `__len__`.

### Приклад 3.6. Межа качиної типізації: метода немає

У партнера немає `eta()`. Поки звіт його не просить — усе добре. Знадобиться найшвидша доставка — помилка.

In [ ]:
def fastest(deliveries):
    return min(deliveries, key=lambda delivery: delivery.eta())

In [ ]:
print(fastest(deliveries).describe())
try:
    fastest(mixed)
except AttributeError as error:
    print("AttributeError:", error)
print([hasattr(delivery, "eta") for delivery in mixed])

Качина типізація — це домовленість: «об'єкт уміє `fare`, `eta`, `describe`». Python перевіряє її лише в момент виклику. `hasattr` показує, хто домовленість порушив. Спільний базовий клас робить таку домовленість явною: якщо нащадок забув метод, він отримає хоча б версію з бази (як `eta()` → 30) або зрозумілий `NotImplementedError`.

---
## 4. Міксин і множинне наслідування

Уночі таксі дорожче на 30 %. Ця надбавка стосується **будь-якої** доставки, тому зручно винести її в окремий маленький клас — **міксин** (mixin). Він не описує самостійну річ, лише додає поведінку. Клас може мати кілька батьків.

**🔮 Прогноз:** скільки коштуватиме нічне таксі на Печерськ (90 грн × 1.3)? Яким буде MRO?

<details>
<summary>Відповідь</summary>

117 грн. MRO: `NightTaxi → NightMixin → TaxiDelivery → Delivery → object`; `super().fare()` у міксині веде до `TaxiDelivery`.

</details>

### Приклад 4.1. `NightMixin`

In [ ]:
class NightMixin:
    """Нічна надбавка 30 % до ціни будь-якої доставки."""

    def fare(self):
        return round(super().fare() * 1.3)

    def describe(self):
        return super().describe() + " (ніч)"


class NightTaxi(NightMixin, TaxiDelivery):
    pass

In [ ]:
night = NightTaxi(5, "Печерськ", "D-2")
print(night.describe())
print([cls.__name__ for cls in NightTaxi.mro()])

Порядок батьків у дужках — порядок MRO. `NightMixin.fare` викликає `super().fare()`, і це **не** `object`: наступний клас у MRO **екземпляра** — `TaxiDelivery`.

```text
NightTaxi.fare()     немає → далі за MRO
NightMixin.fare()    super().fare() × 1.3
   └─ super() ──►  TaxiDelivery.fare()   FEES["Печерськ"] → 90
                    round(90 × 1.3) = 117
```

### Приклад 4.2. Той самий міксин — для кур'єра

Міксин не знає, до якої доставки його підмішають. `super()` щоразу веде до наступного класу в MRO **конкретного** класу.

In [ ]:
class NightCourier(NightMixin, CourierDelivery):
    pass

In [ ]:
night_courier = NightCourier(6, "Поділ")
print(night_courier.describe())
print([cls.__name__ for cls in NightCourier.mro()])

Той самий `super().fare()` у `NightMixin` тепер веде до `CourierDelivery.fare()` → 40, а round(40 × 1.3) = 52. Жодного рядка в міксині не змінилося.

### Приклад 4.3. Порядок батьків має значення

**🔮 Прогноз:** а якщо поставити міксин **другим**: `class LateNightTaxi(TaxiDelivery, NightMixin)`? Скільки коштуватиме Печерськ?

<details>
<summary>Відповідь</summary>

90 грн і без «(ніч)» — міксин не спрацював зовсім. MRO: `LateNightTaxi → TaxiDelivery → Delivery → NightMixin → object`. MRO тримає порядок, у якому записано батьків: уся гілка `TaxiDelivery` (разом з її батьком `Delivery`) іде раніше за `NightMixin`, і міксин опиняється **після** базового класу. А методи `Delivery` не викликають `super()` — до міксина пошук не доходить.

</details>

In [ ]:
class LateNightTaxi(TaxiDelivery, NightMixin):
    pass

In [ ]:
late = LateNightTaxi(5, "Печерськ", "D-2")
print(late.fare(), "|", late.describe())
print([cls.__name__ for cls in LateNightTaxi.mro()])

Помилки немає, просто надбавка тихо зникла — такі баги найважче помітити.

```text
late.fare()       LateNightTaxi → TaxiDelivery.fare → 90         (super() не викликає)
late.describe()   TaxiDelivery.describe
                    └─ super() → Delivery.describe → "…90 грн, ~25 хв"   (super() не викликає)
                  + ", водій D-2"
                  NightMixin стоїть у MRO після Delivery — до нього ніхто не доходить
```

Правило: **міксини — першими в дужках**, основний клас — останнім. Тоді міксин стоїть у MRO раніше за всі методи, які він має обгорнути.

### Приклад 4.4. Два міксини — естафета `super()`

Промокод −10 % — ще один міксин. Підмішаємо обидва. Кожен міксин викликає `super()` і передає естафету далі за MRO.

In [ ]:
class PromoMixin:
    """Знижка 10 % на доставку."""

    def fare(self):
        return round(super().fare() * 0.9)

    def describe(self):
        return super().describe() + " (промо)"


class NightPromoTaxi(PromoMixin, NightMixin, TaxiDelivery):
    pass

In [ ]:
combo = NightPromoTaxi(7, "Печерськ", "D-2")
print(combo.describe())
print([cls.__name__ for cls in NightPromoTaxi.mro()])

```text
PromoMixin.fare()  → super().fare() × 0.9
  NightMixin.fare()  → super().fare() × 1.3
    TaxiDelivery.fare() → 90
                         → round(90 × 1.3) = 117
                         → round(117 × 0.9) = 105
```

Естафета працює, бо **кожен** учасник викликає `super()`. Варто одному не передати виклик далі — і всі класи після нього в MRO мовчки пропускаються. З `__init__` це особливо підступно: атрибути просто не з'являться.

### Задача 1. Обірвана естафета

Міксин `LoggedMixin` пише в журнал `LOG` про створення доставки, але його `__init__` не передає виклик далі. **Прогноз:** що впаде і чому? Допиши `__init__` міксина так, щоб він після запису в журнал передав усі аргументи далі за MRO.

In [ ]:
LOG = []


class LoggedMixin:
    def __init__(self, *args):
        LOG.append(f"створено {type(self).__name__}")
        # YOUR CODE HERE
        pass


class LoggedCourier(LoggedMixin, CourierDelivery):
    pass

In [ ]:
courier = LoggedCourier(8, "Поділ")
print(courier.describe(), LOG)
assert courier.describe() == "Кур'єр №8: Поділ, 40 грн, ~45 хв"
assert LOG == ["створено LoggedCourier"]
try:
    LoggedCourier(9, "Печерськ")
except ValueError:
    pass
else:
    raise AssertionError("перевірка району CourierDelivery має спрацювати і через міксин")
print("✅ Задача 1 розв'язана")

---
## 5. Пастки: несумісні сигнатури й нащадок, що ламає батька

### Приклад 5.1. Несумісні сигнатури

Сервіс створює доставку за назвою типу — таблиця класів, як таблиця команд в уроці 19.

In [ ]:
KINDS = {"taxi": TaxiDelivery, "courier": CourierDelivery, "pickup": Pickup}

In [ ]:
for name, cls in KINDS.items():
    try:
        print(cls(7, "Поділ").describe())
    except TypeError as error:
        print(name, "→", error)

`TaxiDelivery` вимагає більше аргументів, ніж батько, тож код, який створює доставки «за шаблоном батька» `cls(order_id, district)`, ламається лише на ній.

### Приклад 5.2. Сумісний конструктор

Коли класи в ієрархії мають працювати взаємозамінно, їхні конструктори теж мають бути сумісні. Зробимо водія необов'язковим: його можна призначити пізніше.

In [ ]:
class FlexibleTaxi(TaxiDelivery):
    def __init__(self, order_id, district, driver=None):
        super().__init__(order_id, district, driver or "не призначено")


KINDS = {"taxi": FlexibleTaxi, "courier": CourierDelivery, "pickup": Pickup}

In [ ]:
for name, cls in KINDS.items():
    print(cls(7, "Поділ").describe())
print(FlexibleTaxi(8, "Оболонь", "D-3").describe())

Тепер усі три класи можна створити однаково — `cls(7, "Поділ")`, а водія за потреби передати третім аргументом. Таблиця класів працює без жодного `if`.

### Приклад 5.3. Нащадок, що ламає батька

Колега вирішила, що самовивіз має показувати не `0`, а слово.

**🔮 Прогноз:** що станеться зі звітом «Разом за доставку»?

<details>
<summary>Відповідь</summary>

`TypeError`: `sum` не вміє додавати число й рядок.

</details>

In [ ]:
class FriendlyPickup(Pickup):
    def fare(self):
        return "безкоштовно"

In [ ]:
try:
    print(sum(delivery.fare() for delivery in deliveries + [FriendlyPickup(6, "кафе")]))
except TypeError as error:
    print("TypeError:", error)

<details>
<summary>Що тут не так?</summary>

`FriendlyPickup` порушує обіцянку батька «`fare()` повертає число» — принцип підстановки Лісков (LSP). Будь-який код, що рахує суму, падає.

</details>

Перевизначаючи метод, не змінюй, **що** він повертає і **які** винятки піднімає — лише **як** він це обчислює.

### Приклад 5.4. Як зробити правильно

Ціна лишається числом (обіцянка батька), а слово «безкоштовно» з'являється там, де йому місце, — в описі для людини.

In [ ]:
class FriendlyPickup(Pickup):
    def describe(self):
        return f"{self.kind} №{self.order_id}: безкоштовно, ~{self.eta()} хв"

In [ ]:
friendly = FriendlyPickup(6, "кафе")
print(friendly.describe())
print(sum(delivery.fare() for delivery in deliveries + [friendly]))

Звіт знову рахує суму (120 грн — самовивіз додав 0), а клієнт бачить «безкоштовно». Нащадок змінив **представлення**, не порушивши **контракт** `fare()`.

---
## 6. Розібраний приклад: звіт, що не знає типів доставок

### Приклад 6.1. `delivery_report`

In [ ]:
def delivery_report(deliveries):
    by_kind = {}
    for delivery in deliveries:
        by_kind.setdefault(delivery.kind, []).append(delivery)
    lines = []
    for kind, group in by_kind.items():
        total = sum(delivery.fare() for delivery in group)
        lines.append(f"{kind}: {len(group)} шт., {total} грн")
    fastest = min(deliveries, key=lambda delivery: delivery.eta())
    lines.append(f"Найшвидша: {fastest.describe()}")
    return "\n".join(lines)

In [ ]:
shift = [TaxiDelivery(1, "Оболонь", "D-3"), CourierDelivery(2, "Поділ"), Pickup(3, "кафе"),
         NightTaxi(4, "Печерськ", "D-2"), TaxiDelivery(5, "Поділ", "D-1")]
print(delivery_report(shift))

- групи — за атрибутом класу `kind`: кожен підклас визначає свій, а `NightTaxi` успадковує «Таксі» від `TaxiDelivery`;
- `fare()` у кожної доставки своя, і для нічного таксі це 117 з надбавкою: 80 + 117 + 60 = 257. Звіт про надбавку не знає;
- `min` з `key=` з уроку 19 порівнює `eta()` доставок різних класів; `describe()` найшвидшої — теж поліморфний виклик;
- жодного `isinstance` і жодного `if` за типом.

### Приклад 6.2. Новий тип — без змін у звіті

Сервіс додає велосипед: 50 грн, 20 хвилин, будь-який район. Новий клас — і більше нічого.

In [ ]:
class BikeDelivery(Delivery):
    kind = "Велосипед"

    def fare(self):
        return 50

    def eta(self):
        return 20

In [ ]:
print(delivery_report(shift + [BikeDelivery(6, "Печерськ")]))

У звіті з'явився рядок «Велосипед: 1 шт., 50 грн», а `delivery_report` ми не чіпали. Найшвидшою лишився самовивіз (15 хв < 20 хв). Це й є головний виграш поліморфізму: код, що **використовує** доставки, закритий для змін, а набір доставок — відкритий для розширення.

### Задача 2. Доставка дроном

Сервіс тестує **дрон**: `kind = "Дрон"`, 120 грн, 10 хвилин, лише Оболонь (інакше `ValueError` у конструкторі, як у кур'єра). `delivery_report` не змінюй. Нічний дрон `NightDrone(NightMixin, DroneDelivery)` має запрацювати без нового коду в міксині: 156 грн.

Підказка: скопіюй будову `CourierDelivery` — атрибути класу `kind` і `AREA`, перевірка району в `__init__` перед `super().__init__`, `fare()` і `eta()`.

In [ ]:
class DroneDelivery(Delivery):
    # YOUR CODE HERE
    pass


class NightDrone(NightMixin, DroneDelivery):
    pass

In [ ]:
text = delivery_report(shift + [DroneDelivery(6, "Оболонь")])
print(text)
assert "Дрон: 1 шт., 120 грн" in text
assert text.endswith("Найшвидша: Дрон №6: Оболонь, 120 грн, ~10 хв")
assert NightDrone(7, "Оболонь").fare() == 156
try:
    DroneDelivery(9, "Поділ")
except ValueError:
    pass
else:
    raise AssertionError("дрон на Поділ — ValueError")
print("✅ Задача 2 розв'язана")

---
## 7. Архітектура: ієрархія чи композиція?

Задача: три способи доставки, і будь-який може бути нічним. Три рішення:

- **А. Один клас і `if` за типом** (приклад 3.3) — просто, поки способів два; кожен новий спосіб — правки в кожному `if`;
- **Б. Ієрархія класів** — те, що ми зробили в розділах 1–6;
- **В. Композиція** — доставка не **є** нічною чи таксі, вона **має** правило ціни: функцію «район → ціна», яку передають у конструктор.

```text
Б. ієрархія (наслідування: «є»)          В. композиція («має»)

            Delivery                     ComposedDelivery ──має──► pricing: district → ціна
     ┌─────────┼─────────┐                                           ▲
TaxiDelivery Courier... Pickup                      ┌────────────────┼──────────────┐
     │                                        flat(50)       pickup_pricing    promo(pricing, 10)
NightTaxi ◄── NightMixin
```

| | А. `if` за типом | Б. Ієрархія класів | В. Композиція |
|---|---|---|---|
| Новий спосіб доставки | правки в кожному `if` | новий клас | нова функція-правило |
| Комбінації ознак (ніч, промо) | ще більше `if` | вибух класів або міксини | `night(promo(taxi_pricing))` |
| Зв'язність | усе в одному місці | нащадки залежать від батька | частини незалежні |
| Коли доречно | 2 типи, які не зростатимуть | стабільне «є»: самовивіз **є** доставкою | «має», змінні правила, комбінації |

### Приклад 7.1. Вибух класів

Скільки класів потрібно в ієрархії, щоб покрити всі комбінації «спосіб × час × промокод»?

In [ ]:
from itertools import product

METHODS = ["Taxi", "Courier", "Pickup"]
TIMES = ["", "Night"]
PROMOS = ["", "Promo"]

In [ ]:
classes = [f"{time}{promo}{method}" for method, time, promo in product(METHODS, TIMES, PROMOS)]
print(len(classes), classes)
print("частин для композиції:", len(METHODS) + 2)

3 × 2 × 2 = **12** класів (`NightPromoTaxi` з прикладу 4.4 — лише один з них). Додай ще одну ознаку з двома значеннями — 24. У композиції — 3 правила ціни + 2 обгортки (ніч, промо) = 5 частин, які комбінуються як конвеєр з уроку 19.

### Приклад 7.2. Одна доставка — різні правила ціни

Один клас `ComposedDelivery` без підкласів. Ціну рахує не він сам, а передана функція `pricing`. Правила — звичайні функції, а `promo(pricing, percent)` — **фабрика** з уроку 19: бере правило й повертає нове правило зі знижкою.

In [ ]:
class ComposedDelivery:
    def __init__(self, order_id, district, pricing, kind):
        self.order_id = order_id
        self.district = district
        self.pricing = pricing
        self.kind = kind

    def fare(self):
        return self.pricing(self.district)


def pickup_pricing(district):
    return 0


def flat(price):
    """Фабрика: однакова ціна для будь-якого району."""
    def priced(district):
        return price
    return priced


def promo(pricing, percent):
    """Фабрика-обгортка: те саме правило, але зі знижкою percent %."""
    def priced(district):
        return round(pricing(district) * (100 - percent) / 100)
    return priced

In [ ]:
bike = ComposedDelivery(1, "Печерськ", flat(50), "Велосипед")
bike_promo = ComposedDelivery(2, "Поділ", promo(flat(50), 10), "Велосипед")
pick = ComposedDelivery(3, "кафе", pickup_pricing, "Самовивіз")
print(bike.fare(), bike_promo.fare(), pick.fare())
print(type(bike) is type(bike_promo) is type(pick))

Три доставки — один клас. Різниця лише в об'єкті `pricing`, який кожна **має**: `flat(50)` → 50, `promo(flat(50), 10)` → 45, самовивіз → 0. Новий «тип» ціни — нова маленька функція, а не новий клас.

### Приклад 7.3. Змінити правило на льоту

Клієнт ввів промокод уже після оформлення доставки. В ієрархії довелося б створювати об'єкт іншого класу. У композиції — просто замінити правило.

In [ ]:
print("до промокоду:   ", bike.fare())
bike.pricing = promo(bike.pricing, 20)
print("після промокоду:", bike.fare())

50 → 40: той самий об'єкт `bike`, нове правило ціни. Це сильна сторона композиції — поведінку можна збирати й міняти, поки програма працює.

На практиці підходи поєднують: `Delivery` з підкласами для **різних сутностей** (самовивіз **є** доставкою, у таксі **є** водій) і з композицією для **змінних правил** (ніч, промо, акції).

### Задача 3. Те саме композицією

Перепиши ціни таксі й кур'єра для `ComposedDelivery` (приклад 7.2) — без підкласів і без `if` за типом доставки:

- `taxi_pricing(district)` — ціни з `FEES` (приклад 3.3);
- `courier_pricing(district)` — 40 грн, лише Поділ, інакше `ValueError`;
- `night(pricing)` — фабрика, як `promo`: повертає нову функцію, що додає 30 % до ціни `pricing` (з `round`, як `NightMixin`).

Перевірка порівняє ціни з класами з розділу 4.

In [ ]:
# YOUR CODE HERE


In [ ]:
assert ComposedDelivery(1, "Оболонь", taxi_pricing, "Таксі").fare() == 80
assert ComposedDelivery(4, "Печерськ", night(taxi_pricing), "Таксі").fare() == 117
assert ComposedDelivery(2, "Поділ", night(courier_pricing), "Кур'єр").fare() == 52
assert ComposedDelivery(5, "Печерськ", night(taxi_pricing), "Таксі").fare() == NightTaxi(5, "Печерськ", "D-2").fare()
assert ComposedDelivery(6, "Поділ", night(courier_pricing), "Кур'єр").fare() == NightCourier(6, "Поділ").fare()
try:
    ComposedDelivery(7, "Оболонь", courier_pricing, "Кур'єр").fare()
except ValueError:
    pass
else:
    raise AssertionError("кур'єр на Оболонь — ValueError")
print("✅ Задача 3 розв'язана: композиція дає ті самі ціни")

## ✅ Самоперевірка

1. Звідки в `Pickup` береться `describe` і чому `self.fare()` в ньому повертає 0?
2. Що станеться без `super().__init__` у `TaxiDelivery`?
3. Чому `super().fare()` у `NightMixin` веде до `TaxiDelivery`?
4. Чим поганий `FriendlyPickup`?
5. Коли наслідування, а коли композиція?

<details>
<summary>Відповіді</summary>

1. З `Delivery` за MRO; `self` — `Pickup`, тож `fare` шукається з `Pickup`.
2. Не буде `order_id` і `district` — `AttributeError` в `describe`.
3. `super()` — наступний клас у MRO **екземпляра**.
4. Порушує LSP: `fare()` має повертати число.
5. «Є» і стабільно — наслідування; «має», змінні правила й комбінації — композиція.

</details>

### Шпаргалка

```python
class Taxi(Delivery):                        # нащадок
    def __init__(self, order_id, district, driver):
        super().__init__(order_id, district)  # спершу батько
        self.driver = driver
    def fare(self):                          # перевизначення
        return ...
    def describe(self):
        return super().describe() + "..."    # розширення

Taxi.mro()                                   # порядок пошуку методів
class NightTaxi(NightMixin, Taxi): ...       # міксин першим; кожен викликає super()

for d in deliveries:                         # поліморфізм — без if за типом
    d.fare()

Delivery(order_id, district, pricing=night(taxi_pricing))   # композиція
```

## Далі

- Практикум на реальних даних — [`lab_lesson_21_titanic_inheritance.ipynb`](https://github.com/NikoriakViktor/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_2/lessons/lesson_21_inheritance_polymorphism/lab_lesson_21_titanic_inheritance.ipynb): ієрархія пасажирів «Титаніка».
- **Урок 22 — Інкапсуляція, область видимості**: як не дати коду ззовні записати в доставку від'ємну ціну.